# Multimodal Classification

This notebook compares zero-shot CLIP classification and a few-shot linear probe on 10 Food101 classes.

Food101 is treated as having `train` and `validation` splits only. The pipeline further splits `train` into a few-shot train subset and a small dev subset, while the dataset `validation` split is reserved for final evaluation.

Artifacts are saved automatically under `artifacts/CLIP/<timestamp>/`.


In [1]:
from pathlib import Path

req_file = (Path.cwd() / "../../requirements.txt").resolve()
print("Installing from:", req_file)

%pip install -r {req_file}

Installing from: /Users/khoatran/Developer/learning-so-deep/assignments/assignment1/requirements.txt

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()   # .../multimodal
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.food101_experiment import ExperimentConfig, run_experiment

/Users/khoatran/Developer/learning-so-deep/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
config = ExperimentConfig(
    model_id="openai/clip-vit-base-patch32",
    top_k=10,
    class_names=[
        "apple_pie",
        "bibimbap",
        "chicken_wings",
        "donuts",
        "eggs_benedict",
        "french_fries",
        "grilled_cheese_sandwich",
        "hamburger",
        "ice_cream",
        "pizza",
    ],
    shots_per_class=8,
    val_per_class=20,
    batch_size=16,
    epochs=20,
    learning_rate=1e-3,
    weight_decay=1e-4,
    output_root=Path("output/clip-vit-base-patch32"),
)
config


ExperimentConfig(model_id='openai/clip-vit-base-patch32', dataset_name='ethz/food101', top_k=10, class_names=['apple_pie', 'bibimbap', 'chicken_wings', 'donuts', 'eggs_benedict', 'french_fries', 'grilled_cheese_sandwich', 'hamburger', 'ice_cream', 'pizza'], shots_per_class=8, val_per_class=20, batch_size=16, num_workers=0, epochs=20, learning_rate=0.001, weight_decay=0.0001, seed=42, device=None, output_root=PosixPath('output/clip-vit-base-patch32'), prompt_templates=('a photo of {}.', 'a close-up photo of {}.', 'a photo of a plate of {}.', 'a food photo of {}.', 'a restaurant photo of {}.'))

In [4]:
from collections import Counter
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from models import food101_experiment as fexp

# Reuse the same configuration choices as the experiment setup.
eda_config = ExperimentConfig(
    model_id=config.model_id,
    dataset_name=config.dataset_name,
    top_k=config.top_k,
    class_names=config.class_names,
    shots_per_class=config.shots_per_class,
    val_per_class=config.val_per_class,
    seed=config.seed,
)

datasets_by_split, class_names, dataset_summary = fexp.load_food101_datasets(eda_config)
print('Selected classes:', [name.replace('_', ' ') for name in class_names])
print('Dataset summary:', dataset_summary)

rows = []
for split_name, split_dataset in datasets_by_split.items():
    label_ids = [split_dataset.original_to_new[label] for label in split_dataset.dataset_split['label']]
    counts = Counter(label_ids)

    split_counts = [counts.get(class_id, 0) for class_id in range(len(class_names))]
    min_count = min(split_counts) if split_counts else 0
    max_count = max(split_counts) if split_counts else 0
    balanced = len(set(split_counts)) == 1

    print(f"\n[{split_name}] min={min_count}, max={max_count}, balanced={balanced}")
    for class_id, count in enumerate(split_counts):
        rows.append(
            {
                'split': split_name,
                'class_name': class_names[class_id],
                'count': count,
            }
        )

distribution_df = pd.DataFrame(rows)
pivot_df = distribution_df.pivot(index='class_name', columns='split', values='count').fillna(0).astype(int)
display(pivot_df.sort_index())

plt.figure(figsize=(12, 5))
sns.barplot(data=distribution_df, x='class_name', y='count', hue='split')
plt.xticks(rotation=45, ha='right')
plt.title('Food101 Class Distribution by Split')
plt.tight_layout()
plt.show()

# Simple numeric imbalance score: (max-min)/max in each split (0 means perfectly equal).
imbalance_by_split = (
    distribution_df.groupby('split')['count']
    .agg(['min', 'max'])
    .assign(imbalance_ratio=lambda d: (d['max'] - d['min']) / d['max'].clip(lower=1))
)
display(imbalance_by_split)

Selected classes: ['apple pie', 'bibimbap', 'chicken wings', 'donuts', 'eggs benedict', 'french fries', 'grilled cheese sandwich', 'hamburger', 'ice cream', 'pizza']
Dataset summary: {'dataset_name': 'ethz/food101', 'available_splits': ['train', 'validation'], 'selected_class_names': ['apple_pie', 'bibimbap', 'chicken_wings', 'donuts', 'eggs_benedict', 'french_fries', 'grilled_cheese_sandwich', 'hamburger', 'ice_cream', 'pizza'], 'selected_class_names_readable': ['apple pie', 'bibimbap', 'chicken wings', 'donuts', 'eggs benedict', 'french fries', 'grilled cheese sandwich', 'hamburger', 'ice cream', 'pizza'], 'shots_per_class': 8, 'dev_per_class': 20, 'few_shot_train_size': 80, 'few_shot_dev_size': 200, 'held_out_validation_size': 2500, 'few_shot_train_size_per_class': 8, 'few_shot_dev_size_per_class': 20, 'evaluation_split_name': 'validation'}

[few_shot_train] min=8, max=8, balanced=True

[few_shot_dev] min=20, max=20, balanced=True

[validation] min=250, max=250, balanced=True


split,few_shot_dev,few_shot_train,validation
class_name,,,
apple_pie,20,8,250
bibimbap,20,8,250
chicken_wings,20,8,250
donuts,20,8,250
eggs_benedict,20,8,250
french_fries,20,8,250
grilled_cheese_sandwich,20,8,250
hamburger,20,8,250
ice_cream,20,8,250


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_46919/3031645624.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,min,max,imbalance_ratio
split,,,
few_shot_dev,20,20,0.0
few_shot_train,8,8,0.0
validation,250,250,0.0


In [7]:
results = run_experiment(config)
results


Loading dataset 'ethz/food101'...
Loading CLIP model 'openai/clip-vit-base-patch32' on mps...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 56278.50it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building dataloaders...
Extracting image features...


/Users/khoatran/Developer/learning-so-deep/venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Building zero-shot classifier...
Training few-shot linear probe...
Artifacts saved to output/clip-vit-base-patch32/20260326_001759
Zero-shot accuracy: 0.9792 | Few-shot accuracy: 0.8916


{'output_dir': 'output/clip-vit-base-patch32/20260326_001759',
 'dataset_summary': {'dataset_name': 'ethz/food101',
  'available_splits': ['train', 'validation'],
  'selected_class_names': ['apple_pie',
   'bibimbap',
   'chicken_wings',
   'donuts',
   'eggs_benedict',
   'french_fries',
   'grilled_cheese_sandwich',
   'hamburger',
   'ice_cream',
   'pizza'],
  'selected_class_names_readable': ['apple pie',
   'bibimbap',
   'chicken wings',
   'donuts',
   'eggs benedict',
   'french fries',
   'grilled cheese sandwich',
   'hamburger',
   'ice cream',
   'pizza'],
  'shots_per_class': 8,
  'dev_per_class': 20,
  'few_shot_train_size': 80,
  'few_shot_dev_size': 200,
  'held_out_validation_size': 2500,
  'few_shot_train_size_per_class': 8,
  'few_shot_dev_size_per_class': 20,
  'evaluation_split_name': 'validation',
  'device': 'mps'},
 'comparison_summary': {'class_names': ['apple_pie',
   'bibimbap',
   'chicken_wings',
   'donuts',
   'eggs_benedict',
   'french_fries',
   'gril

The run directory contains:

- `config.json` and `dataset_summary.json`
- `zero_shot/` metrics, predictions, confusion matrix CSV/PNG on the held-out `validation` split
- `few_shot_linear_probe/` metrics, predictions, confusion matrix CSV/PNG on the held-out `validation` split
- `few_shot_linear_probe.pt` linear-probe weights
- `few_shot_training_history.csv`, `comparison.csv`, `comparison_summary.json`, and `run_summary.json`
